# Import de dados e bibliotecas para tratamento incial e criação das variáveis

In [1]:
import numpy as np
import polars as pl
import pyarrow
import gc

# Tratamentos da base de apontamentos

In [2]:
df_apontamentos = pl.read_parquet('datasets/apontamentos/desenvolver_apontamentos.parquet')
df_apontamentos

Id,Inicio,Fim,Tag,Frota,Tipo,Classe
i64,datetime[ns],datetime[ns],str,str,str,str
23462554,2025-01-04 19:12:54,2025-01-04 19:40:29,"""CA65789""","""793-D 2S""","""Caminhao""","""Operando"""
23462555,2025-01-04 19:12:28,2025-01-04 19:38:15,"""CA65908""","""793-D 3S""","""Caminhao""","""Operando"""
23462556,2025-01-04 19:14:07,2025-01-04 19:18:29,"""CA65915""","""793-D 4S""","""Caminhao""","""Parado"""
23462759,2025-01-04 19:20:36,2025-01-04 19:27:44,"""CA5926""","""793-D 5S""","""Caminhao""","""Parado"""
23462762,2025-01-04 19:18:29,2025-01-04 19:23:51,"""CA65915""","""793-D 4S""","""Caminhao""","""Operando"""
…,…,…,…,…,…,…
197055043,2025-01-18 01:00:00,2025-01-18 02:00:00,"""PE3782""","""LeTourneau L 1850""","""Escavadeira""","""Hibernando"""
197055044,2025-01-18 02:00:00,2025-01-18 03:00:00,"""PE3782""","""LeTourneau L 1850""","""Escavadeira""","""Hibernando"""
197055045,2025-01-18 03:00:00,2025-01-18 04:00:00,"""PE3782""","""LeTourneau L 1850""","""Escavadeira""","""Hibernando"""


In [3]:
df_apontamentos = df_apontamentos.sort(["Tag", "Inicio"])

# Cria a coluna 'bloco_id' que será usada para o agrupamento
df_apontamentos = df_apontamentos.with_columns(
    (
        (pl.col("Tag") != pl.col("Tag").shift(1)) |
        (pl.col("Classe") != pl.col("Classe").shift(1))
    )
    .fill_null(True) # O primeiro shift gera nulo, então forçamos True para iniciar o bloco 1
    .cum_sum()
    .alias("bloco_id")
)

# Agrupa com base na coluna e extrai os horarios
df_apontamentos_agg = (
    df_apontamentos.group_by(["Tag", "Frota", "Tipo", "Classe", "bloco_id"])
    .agg([
        pl.col("Inicio").min().alias("Inicio"),
        pl.col("Fim").max().alias("Fim")
    ])
    # 4. Limpeza e ordenação final
    .drop("bloco_id")
    .sort(["Tag", "Inicio"])
)

In [4]:
df_apontamentos_agg = df_apontamentos_agg.rename({"Tag": "TAG"})

In [5]:
df_apontamentos_agg.write_parquet('datasets/datasets_tratados/apontamentos_agrupados.parquet')

# Tratamentos da base de telemetrias

In [6]:
plan_telemetria = pl.scan_parquet(['datasets/telemetria/telemetry_jan.parquet',
                                 'datasets/telemetria/telemetry_feb.parquet',
                                 'datasets/telemetria/telemetry_mar.parquet',
                                 'datasets/telemetria/telemetry_abr.parquet',
                                 'datasets/telemetria/telemetry_may.parquet',
                                 'datasets/telemetria/telemetry_jun.parquet'])

In [7]:
# Limpando colunas que não serão uteis para as analises

cols_para_remover = [
    'Id_Eventos_Telemetria', 
    'Localidade', 
    'Matricula_Operador_Hash', 
    'Id_Alarme', 
    'Id_Criticidade'
]

plan_telemetria = plan_telemetria.drop(cols_para_remover).with_columns([
    pl.col("Data_Evento").cast(pl.Datetime("us")),
    pl.col("Dia").cast(pl.Int8)
])

In [8]:
plan_telemetria

# Junção das bases

In [9]:
df_apontamentos_agg = df_apontamentos_agg.with_columns(
    pl.col("Inicio").cast(pl.Datetime("us"))
).sort(["TAG", "Inicio"])

# O plano de telemetria precisa estar ordenado por Data_Evento para o join_asof
plan_telemetria = plan_telemetria.sort("Data_Evento") 

# Executa o join de forma Lazy
plan_conjunto = plan_telemetria.join_asof(
    df_apontamentos_agg.lazy(), # transforma em lazy temporariamente para o plano
    left_on="Data_Evento",
    right_on="Inicio",
    by="TAG",
    strategy="backward"
)

In [10]:
del df_apontamentos_agg
gc.collect()

13

# Engenharia de features na base conjunta

In [11]:
# Unificando as transformações de engenharia de features
mapa_substituicao = {
    "Critico": "Crítico",
    "N??o Crítico": "Não Crítico",
    "Não Cr??tico": "Não Crítico"
}

plan_conjunto = plan_conjunto.with_columns([
    
    # Tratamento da string de Turno
    pl.col("Inicio_Turno")
      .str.to_datetime("%Y-%m-%d %H:%M:%S.%3f")
      .alias("Inicio_Turno_dt"),
    
    # Tratamento e conversão da coluna Valor de forma segura
    pl.col("Valor")
      .str.replace_all(",", ".", literal=True)
      .cast(pl.Float64, strict=False),

    pl.col("Criticidade").replace(mapa_substituicao),

    pl.col("Alarme").str.to_lowercase()   
    
]).with_columns([
    
    # Criando colunas dependentes do Inicio_Turno_dt
    pl.when(pl.col("Inicio_Turno_dt").dt.hour() == 18).then(pl.lit("Turno 18-6"))
      .when(pl.col("Inicio_Turno_dt").dt.hour() == 6).then(pl.lit("Turno 6-18"))
      .otherwise(pl.lit(None)).alias("Turno"),
      
    (pl.col('Data_Evento') - pl.col('Inicio')).dt.total_seconds().alias("Tempo_Situacao"),
    (pl.col('Data_Evento') - pl.col('Inicio_Turno_dt')).dt.total_seconds().alias("Tempo_Turno"),
    
    pl.col('Data_Evento').dt.month().alias("Mes")
    
]).drop([
    'Inicio_Turno', 'Inicio_Turno_dt', 'Fim_Turno', 'Inicio', 'Fim', 'Tipo_right', 'Frota'
    
]).rename(
    {"Classe_right": "Situacao_Operacional"}
)

In [12]:
# Filtrando e salvando Caminhão via Streaming
plan_conjunto.filter(pl.col("Tipo") == 'Caminhao').sink_parquet(
    'datasets/datasets_tratados/df_caminhao.parquet', 
    engine='streaming'
)

# Filtrando e salvando Escavadeira via Streaming 
plan_conjunto.filter(pl.col("Tipo") == 'Escavadeira').sink_parquet(
    'datasets/datasets_tratados/df_escavadeira.parquet', 
    engine='streaming'
)

/tmp/ipykernel_17616/4115280003.py:2: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  plan_conjunto.filter(pl.col("Tipo") == 'Caminhao').sink_parquet(
/tmp/ipykernel_17616/4115280003.py:8: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  plan_conjunto.filter(pl.col("Tipo") == 'Escavadeira').sink_parquet(
